In [ ]:
# pip install uv mcp nest-asyncio

  Using cached mcp-1.28.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pydantic_settings-2.14.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached pyjwt-2.13.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached uvicorn-0.51.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-2026.6.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached cryptography-49.0.0-cp311-abi3-macosx_11_0_arm64.whl.metadata (4.3 kB)
  Using cached cffi-2.1.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.5 k

In [3]:
pip show uv

Name: uv
Version: 0.11.31
Summary: An extremely fast Python package and project manager, written in Rust.
Home-page: https://pypi.org/project/uv/
Author: 
Author-email: "Astral Software Inc." <hey@astral.sh>
License-Expression: MIT OR Apache-2.0
Location: /opt/miniconda3/envs/workshop2_env/lib/python3.12/site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
!npm -v

11.9.0


In [5]:
!node -v

v24.14.0


# Creating an MCP Server

In this lesson, you will wrap the tools of the chatbot of the previous lesson, to build an MCP server that exposes 5 workflow tools. You will use here the `stdio` transport and run the server in your local environment.

## How can you create an MCP Server? - *Additional Note*

Let's take the example of a server that exposes tools. This server needs to handle two main requests from the client:
- listing all the tools
  
   <img src="images/server_list_tools.png" width="400">

- executing a particular tool
  
  <img src="images/server_call_tool.png" width="400">

There are two ways for creating an MCP server:
- **low-level implementation**: in this approach, you directly define and handle the various types of requests (`ListToolsRequest` and `CallToolRequest`). This approach allows you to customize every aspect of your server.
- **high-level implementation using `FastMCP`**: `FastMCP` is a high-level interface that makes building MCP servers faster and simpler. In this approach, you just focus on defining the tools as functions, and`FastMCP` handles all the protocol details.
  
You will use in this exercise `FastMCP`.

There are two ways for creating an MCP server:
- **low-level implementation**: in this approach, you directly define and handle the various types of requests (`ListToolsRequest` and `CallToolRequest`). This approach allows you to customize every aspect of your server.
- **high-level implementation using `FastMCP`**: `FastMCP` is a high-level interface that makes building MCP servers faster and simpler. In this approach, you just focus on defining the tools as functions, and`FastMCP` handles all the protocol details.
  
You will use in this exercise `FastMCP`.

### Before anything else

- First make sure you have `uv` installed. You can check the installation methods [here](https://docs.astral.sh/uv/getting-started/installation/#pypi).
- Second you need to make sure you have `Node.js` installed on your machine. This is needed to run packages written in  `Typescript` (like the MCP inspector and maybe other MCP servers that you may download to use in your app). For instructions, check [here](https://docs.npmjs.com/downloading-and-installing-node-js-and-npm)

## Building your MCP Server using `FastMCP`

<hr>
<h4 style="color:green; font-weight:bold;">TIPS:</h4>

* This exercise is a follow-along. 

* You can add new cells to experiment
 
<hr>

You will build the files needed for your MCP server. You will first create the python file of the server `onelab_server.py`, then you'll prepare the environment to run the server.

##### Cell 1 (No need to run this cell)

In [ ]:
#%%writefile onelab_server/onelab_server.py
from concurrent.futures import ThreadPoolExecutor
from typing import Tuple
from instructions_and_templates import db_schema
from mcp.server.fastmcp import FastMCP
from utils import (
    html_visualization,
    extract_values,
    generate_sql,
    execute_sql,
    extract_rag_context,
    run_rag,
    get_waypoints,
    geocode_loc,
    parse_locations,
    get_agency_coord,
    generate_summary,
    parse_nearest_labs_info,
    nearest_labs,
    generate_code,
    execute_python,
    final_response,
)

mcp = FastMCP("onelab_server")

@mcp.tool()
def txt2sql_workflow(user_query: str) -> Tuple[str,str]:
    """
    Query the OneLab.db using natural language input.
    Answers queries on information on the location of laboratories and agencies as well as the services that they offer.

    Args:
        user_query: natural language query

    Returns:
        response: textual summary of the retrieved rows
        final_html: html map visualization of the retrieved rows is string format
    """
    value_match_string = extract_values(user_query=user_query)
    sql = generate_sql(user_query=user_query, value_match_string=value_match_string, db_schema=db_schema)
    rows = execute_sql(sql=sql)
    final_html = html_visualization(rows=rows)
    response = generate_summary(user_query=user_query, rows=rows)
    return response, final_html

@mcp.tool()
def rag_workflow(user_query: str) -> Tuple[str,None]:
    """
    Perform RAG on available citizen's charter and service catalog files
    Provides information on client steps, processes, requirements, or information on how to avail of a particular service in an agency
    
    Args:
            user_query: natural language query
    
    Returns:
            response: textual response grounded on available related documents
    """
    result = extract_rag_context(user_query=user_query, sanity_check=False)
    response = run_rag(agencyname=result['agency'], testname=result['testname'], pages=result['pages'], method=result['method'], reference=result['reference'], fee=str(result['fee']), sanity_check=False)
    return response, None

@mcp.tool()
def waypoints_workflow(user_query:str) -> Tuple[str,str]:
    """
    Uses google's directions API to provide waypoints
    Answers questions on how to go to a particular agency or laboratory from a certain location

    Args:
        user_query: natural language query

    Returns:
        response: textual summary of the retrieved rows
        final_html: html map visualization of the retrieved rows is string format
    """
    locations = parse_locations(user_query)
    agency = locations.agency
    user_loc = locations.user_location

    with ThreadPoolExecutor(max_workers=2) as executor:
        destination_future = executor.submit(get_agency_coord, agency)
        origin_future = executor.submit(geocode_loc, user_loc)

        destination = destination_future.result()
        origin = origin_future.result()

    waypoints = get_waypoints(origin['lat'], origin['lng'], destination['lat'], destination['lng'])
    final_html = html_visualization(waypoints=waypoints)
    response = generate_summary(user_query=user_query, waypoints=waypoints)
    return response, final_html

@mcp.tool()
def nearest_labs_workflow(user_query: str) -> Tuple[str,str]:   
    """
    Uses and executes an SQLite template to provide infromation on nearest laboratories and/or the services that they offer given a reference location

    Args:
        user_query: natural language query

    Returns:
        response: textual summary of the retrieved rows
        final_html: html map visualization of the retrieved rows is string format    
    """ 
    response = parse_nearest_labs_info(user_query)
    print(response)
    kwargs = {}
    if response.reference_location:
        reference_location_coord = geocode_loc(response.reference_location)
        lat = reference_location_coord['lat']
        lng = reference_location_coord['lng']
        kwargs["lat"]=lat
        kwargs["lng"]=lng
    if response.limit:
        kwargs["limit"] = response.limit
    if response.testname:
        kwargs["testname"] = response.testname
    print(kwargs)
    rows  = nearest_labs(**kwargs)
    final_html = html_visualization(rows=rows)
    response = generate_summary(user_query=user_query, rows=rows)
    return response, final_html

@mcp.tool()
def analysis_workflow(user_query:str) -> Tuple[str,None]:
    """
    Performs code execution to access csv data.
    Provides answers to hotspot and service area analysis questions (data is for provincial level analysis only)

    Args:
        user_query: natural language query

    Returns:
        response: textual summary of the retrieved data
    """
    code = generate_code(user_query)
    output = execute_python(code)
    response = final_response(user_query=user_query, output=output)
    return response, None

if __name__=="__main__":
    mcp.run(transport='stdio')


Overwriting onelab_server/onelab_server.py


## Setting up your Environment & Testing your Server

You'll now set up the environment that you will use to run and test the server. For that, you will use the `uv` tool, which helps you manage your Python environment: it automatically sets up the project files and manages the package dependencies.

**Terminal Instructions**

- Open a terminal in vscode.
- Navigate to the project directory and initiate it with `uv`:
    - `cd onelab_server`
    - `uv init`
-  Create virtual environment and activate it:
    - `uv venv`
    - `source .venv/bin/activate`
- Install dependencies:
    - `uv add mcp openai rank-bm25 pandas python-dotenv sqlalchemy pypdf ipython requests`
- Launch the inspector:
    - `npx @modelcontextprotocol/inspector uv run onelab_server.py`
- You will get a message saying that the inspector is up and running at a specific address. To open the inspector, click on that given address. The inspector will open in another tab.
- Once you're done with the inspector UI, make sure to close the inspector by typing `Ctrl+C` in the terminal below.

# Creating an MCP Client 

In the previous exercise, you created an MCP onelab server that exposes 5 workflow tools. In this exercise, you will make the chatbot communicate to the server through an MCP client. This will make the chatbot MCP compatible. You will continue from where you left off.

## Back to the Chatbot Example

Here are the main code parts (`process_query`, `chat_loop`) from the chatbot example of exercise 1. Notice that the burden of tool definitions and execution is now shifted onto the MCP server, so the chatbot logic only contains code related to processing the user queries and to keeping the chat loop running until the user types `quit`.

##### Cell 2 (No need to run this cell)

In [ ]:
#%%writefile onelab_chatbot/mcp_chatbot.py

from utils import is_threat
from openai import OpenAI
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal, Optional, List
from datetime import datetime
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
import asyncio
import nest_asyncio
import json
from pathlib import Path
import webbrowser

nest_asyncio.apply()

load_dotenv()

client = OpenAI(
    api_key=os.getenv("BEDROCK_KEY"),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1"
)

class WorkflowChoice(BaseModel):
    choice: Optional[Literal[
        "txt2sql_workflow",
        "rag_workflow",
        "waypoints_workflow",
        "nearest_labs_workflow",
        "analysis_workflow"        
    ]] = Field(
        default = None,
        description = """`txt2sql_workflow`: the user is asking for information on the location of laboratories and agencies as well as the services that they offer.
        `waypoints_workflow`: the user is asking about how to go to a particular agency from a certain location.
        `rag_workflow`: the user is asking for client steps, processes, requirements, or information on how to avail of a particular service in an agency
        `nearest_labs_workflow`: the user is asking for nearest laboratories and/or the services that they offer given a reference location
        `analysis_workflow`: the user is asking about hotspot and service area analysis questions (data is for provincial level analysis only)
        """
    )
    fallback: Optional[str] = Field(default=None, description="Fallback response if the user's query does not fall in any of the given choices")

system_prompt = """Your task is to choose the appropriate workflow depending on the user's intent.
`txt2sql_workflow`: the user is asking for information on the location of laboratories and agencies as well as the services that they offer.
`waypoints_workflow`: the user is asking about how to go to a particular agency from a certain location.
`rag_workflow`: the user is asking for client steps, processes, requirements, or information on how to avail of a particular service in an agency
`nearest_labs_workflow`: the user is asking for nearest laboratories and/or the services that they offer given a reference location
`analysis_workflow`: the user is asking about hotspot and service area analysis questions

For more context, these are some of the queries that can be processed by the workflows:
`What services does DOST-ITDI offer?`
`What do I need to prepare for pipe stiffness test for pvc in DOST-ITDI`
`How do I get to DOST-ASTI from SMDC Light Residences`
`10 nearest laboratories to SMDC Light Residences that offer coliform count`
`Which provinces are classified as potentially underserved`

If the user's intent is does not fall in any of the workflows, return a fallback reply highlighting allowed questions.
"""

class MCP_ChatBot:

    def __init__(self):
        self.session: ClientSession = None
        self.available_tools: List[dict] = []

    async def get_intent(self, user_query):
        response = client.responses.parse(
            model="openai.gpt-5.6-luna",
            input = [
                {
                    "role":"system",
                    "content": system_prompt
                },
                {
                    "role":"user",
                    "content": user_query
                }
            ],
            text_format = WorkflowChoice
        )
        return response.output_parsed

    async def chat(self):
        if datetime.now().hour < 12:
            time = "morning"
        elif datetime.now().hour >= 12:
            time = "afternoon"
        else:
            time = "evening"
        print(f"\nOneLab Agent:\nGood {time}! How may I assist you?")

        while True:

            user_query = input("Enter your query: ")
            print(f"\nUser:\n{user_query}")

            if user_query.lower() == "quit":
                break

            _is_threat, errors = is_threat(user_query)

            if _is_threat:
                print("\nOneLab Agent:\nI'm sorry, but I couldn't process your request as it was potentially unsafe. If you think this was a mistake, please try rephrasing your prompt or providing more context so I can better understand your request.")
                continue

            response = await self.get_intent(user_query)
            workflow = response.choice
            fallback = response.fallback

            if workflow:
                response = await self.session.call_tool(workflow, arguments = {"user_query":user_query})
                #result, html = response.structuredContent["result"]
                if response.isError:
                    print(response.content[0].text)
                    return

                if response.structuredContent is not None:
                    result, html = response.structuredContent["result"]
                else:
                    # Fallback to TextContent
                    texts = [c.text for c in response.content]

                    result = texts[0] if len(texts) > 0 else None
                    html = texts[1] if len(texts) > 1 else None
                print(f"\nOneLab Agent:\n{result}")
                if html:
                    print("---\nOpen `output.html` for the visualization.")
                    webbrowser.open(Path("output.html").resolve().as_uri())
            elif fallback:
                print(f"\nOneLab Agent:\n{fallback}")

    async def connect_to_server_and_run(self):
        # Create server parameters for stdio connection
        server_params = StdioServerParameters(
            command="uv",  # Executable
            args=["run", "../onelab_server/onelab_server.py"],  # Optional command line arguments
            env=None,  # Optional environment variables
        )
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                self.session = session
                # Initialize the connection
                await session.initialize()
    
                # List available tools
                response = await session.list_tools()
                
                tools = response.tools
                print("\nConnected to server with tools:", [tool.name for tool in tools])
                
                self.available_tools = [{
                    "type": "function",
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema
                } for tool in response.tools]
    
                await self.chat()

async def main():
    chatbot = MCP_ChatBot()
    await chatbot.connect_to_server_and_run()
  

if __name__ == "__main__":
    asyncio.run(main())

Overwriting onelab_chatbot/mcp_chatbot.py


## Running the MCP Chatbot

**Terminal Instructions**

- Open a terminal
- Navigate to the `onelab_chatbot` directory:
    - `cd onelab_chatbot`
    - `uv init`
- Activate the virtual environment:
    - `uv venv`
    - `source .venv/bin/activate`
- Install the additional dependencies:
    - `uv add mcp nest-asyncio openai rank-bm25 pandas python-dotenv sqlalchemy pypdf ipython requests`
- Run the chatbot:
    - `uv run mcp_chatbot.py`
- To exit the chatbot, type `quit`.